In [155]:
import pandas as pd
import numpy as np
from datetime import datetime
import sys
import warnings
import nest_asyncio
nest_asyncio.apply()  # 중첩 이벤트 루프 허용
import os
import json
import logging
import asyncio
import pandas as pd
import openai
from datetime import datetime
import sys
from typing import List, Dict, Optional
from tenacity import retry, stop_after_attempt, wait_exponential
import re

warnings.simplefilter(action='ignore', category=FutureWarning) # FutureWarning 제거

In [156]:
df = pd.read_excel('../../data/centum_data/21.11-24.6환자 CC_PI_치료계획.xlsx')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df.iloc[:,1:]

df.columns = df.columns.str.strip()
df = df[['환자번호', '날짜', 
        # 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI',  # 처리 됨.
        'CMO','MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
        'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
        'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt',
        'Lateral excursion Protrusive excursion', 'End feel', 
    #    '치료계획', # 필요 없을 듯.
    #    'T-scan 악화/개선', 'CBCT 악화/개선', 'CBCT 판독소견' # 고유겂 NaN    
    ]]

df = df[['환자번호', '날짜',
        'CMO','MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
        'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
        'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt',
        'Lateral excursion Protrusive excursion', 'End feel']]

In [157]:
df.columns

Index(['환자번호', '날짜', 'CMO', 'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading',
       'Occlusion', 'OJ/OB', 'Class', 'Midline Shift', 'Deviation', 'CR-CO',
       'Tongue ridging', 'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt',
       'Lateral excursion Protrusive excursion', 'End feel'],
      dtype='object')

### CMO & MMO
CMO : 입을 편하게 벌릴 수 있는 최대 크기 \
MMO : 입을 아프더라도 벌릴 수 있는 최대 크기
- 스킴
    - 벌려지는 정도 & 위치 & 처방 종류
    - 화살표 이후의 정보 삭제
    - 고착, 스프레이 필요 없음


In [158]:
df.sample(10).MMO


4938                   35mm --> mm after spray and stretch
22278         45mm --> mm after spray and stretch, no pain
15422     42mm (RT M 당기는 느낌)--> mm after spray and stretch
18272                               48mm Lt) pain, Popping
25549    43mm -->48 mm after spray and stretch and mani...
26014                  33mm --> mm after spray and stretch
14464                  48mm --> mm after spray and stretch
20934               37 mm --> 44mm after spray and stretch
16571                  55mm --> mm after spray and stretch
9864                 45mm --> 50mm after spray and stretch
Name: MMO, dtype: object

In [159]:
df.sample(10).CMO

650       36mm --> mm after spray and stretch
9634      43mm --> mm after spray and stretch
456      30mm --> mm after spray and  stretch
5577      50mm --> mm after spray and stretch
20185                                    46mm
13370                                     NaN
15479                                     NaN
21996     50mm --> mm after spray and stretch
485       mm --> 50mm after spray and stretch
11753     43mm --> mm after spray and stretch
Name: CMO, dtype: object

In [160]:
def extract_number(text):
    """
    텍스트에서 숫자를 추출하는 함수
    NaN 값이나 None 값도 처리 가능
    """
    # NaN, None 또는 비문자열 타입 처리
    if pd.isna(text) or text is None or not isinstance(text, str):
        return '', ''
    
    # 패턴 1: "숫자 mm --> 숫자 mm" 형태 (공백 허용)
    pattern1 = r'(\d+)\s*mm\s*-->\s*(\d+)\s*mm'
    match1 = re.search(pattern1, text)
    
    if match1:
        return match1.group(1), match1.group(2)  # 앞, 뒤 숫자 모두 반환
    
    # 패턴 2: "숫자 mm --> mm" 형태 (뒤 숫자 없음, 공백 허용)
    pattern2 = r'(\d+)\s*mm\s*-->\s*mm'
    match2 = re.search(pattern2, text)
    
    if match2:
        return match2.group(1), ''  # 앞 숫자만 반환, 뒤는 빈 문자열
    
    # 패턴 3: 단순히 "숫자 mm" 형태 (공백 허용)
    pattern3 = r'(\d+)\s*mm'
    match3 = re.search(pattern3, text)
    
    if match3:
        return match3.group(1), ''  # 앞 숫자만 반환, 뒤는 빈 문자열
    
    return '', ''  # 숫자를 찾지 못한 경우 빈 문자열 두 개 반환

In [161]:
# CMO
result = df.CMO.apply(extract_number)
df['CMO_before'], df['CMO_after'] = zip(*result)
# MMO
result = df.MMO.apply(extract_number)
df['MMO_before'], df['MMO_after'] = zip(*result)

### Deviation
입을 벌리면서 위 턱에 대해 아래 턱이 어떻게 움직이는지 경로를 나타내는 열

- 스킴
    - 방향 & 이동 경로 & 치우침의 강도 or 증상
    - L이 가장 안좋음. S는 낫배드.
    - L, S가 중요하며, 강도는 크게 중요하지 않음.
    고착 후 등 다 필요 없다.

In [9]:
df.Deviation.unique()



array([nan, 'L (Lt)', '-', 'R', 'S', '왼L', '왼 L', 'RT) L', '오른쪽 S',
       '오른쪽 S, 벌리고 다물때', 'Rt)L', 'Lt) L', 'RT', '오 L', '오 S', 'Rt', '오L',
       'Lt', 'Lt L 심함', '오', '오른쪽', 'LT', 'L', 'Lt) S', 'Rt) L',
       '안풀고도 심함,풀어서 L 심함', 'S로 약하게', 'S (Rt)', 'Lt 많이 좋아짐', '오른쪽L',
       '오른쪽L /고착 후  오L없어지고 S로 바뀜', 'Lt)L', 'RT)L 약간', 'RT)L',
       'mild L deviation', 's', '없음', 'Lt L', '오른쪽 L', 'n/s', 'Rt) S 거칠게',
       'Rt) S ->거의없음', 'Rt)S', 'Lt S', 'Rt L', 'Rt) L 의심', 'Rt) S',
       'Lt L -> S after spray', 'L왼', '왼L심함', '오s', 'S 심함', 'L,S 중간정도',
       'S(Rt)', 's 심함', '약간 Rt', '오른쪽 L 약간', 'Rt) L 마지막에 원래대로 돌아옴',
       'LT) L', 'Lt)L 있었던적 있음', 'Lt) S,L 중간', 'S (왼쪽으로 갔다가 가운데)', '오l',
       's심함', 'RT L', '왼L++', '안틀면 오L', '왼쪽L', '왼쪽l', '왼쪽 L 거의없음', 'L 살짝',
       'Z', 'both L', '약간 s', 'R 많이 기울어짐', '오S', 'S심함', '왼S',
       'L->S(고착 후)', 'S 약하게', '왼L (양쪽 걸린 느낌)', '양쪽L',
       'S 벌릴때 오른쪽->다물때 왼쪽->', '오L+S중간', '오l경향성', '살짝 L', 'S, L 중간',
       'S (가끔씩 Rt L)', 'R(S)', 'R(L)', 'L(L)', 'R

In [175]:
import re
import pandas as pd
from collections import Counter

def extract_pattern_info(text):
    """
    Extract pattern type, direction, and intensity from a single deviation description.
    
    Args:
        text (str): A deviation description string
        
    Returns:
        tuple: (pattern_type, direction, intensity)
    """
    # Initialize values
    pattern_type = []
    direction = 'unspecified'
    intensity = 'normal'
    
    # Skip NaN values
    if not isinstance(text, str) or text.lower() == 'nan':
        return ('other', direction, intensity)
    
    # Extract pattern type (L, S)
    if re.search(r'(?i)(?:^|\s|[,(])L(?:$|\s|[,)]|t\)|\()', text):
        pattern_type.append('L')
    if re.search(r'(?i)(?:^|\s|[,(])S(?:$|\s|[,)]|t\)|\()', text):
        pattern_type.append('S')
    
    # Extract direction
    left_match = re.search(r'(?i)(?:왼|왼쪽|lt|LT)', text)
    right_match = re.search(r'(?i)(?:오|오른|오른쪽|rt|RT)', text)
    
    if left_match and right_match:
        direction = 'both'
    elif left_match:
        direction = 'left'
    elif right_match:
        direction = 'right'
    
    # Extract intensity
    if re.search(r'(?i)(?:심함|많이|\+\+)', text):
        intensity = 'severe'
    elif re.search(r'(?i)(?:약간|약하게)', text):
        intensity = 'mild'
    elif re.search(r'(?i)(?:거의 없음|없음)', text):
        intensity = 'none'
    elif re.search(r'(?i)(?:중간|중간정도)', text):
        intensity = 'moderate'
        
    # Convert pattern_type list to string
    pattern_type = '+'.join(pattern_type) if pattern_type else 'other'
    
    return (pattern_type, direction, intensity)

def analyze_deviation_patterns(df, col_name='Deviation'):
    """
    Analyze deviation patterns in a dataframe column and add new columns for pattern type, 
    direction, and intensity.
    
    Args:
        df (pandas.DataFrame): DataFrame containing the deviation column
        col_name (str): Name of the column containing deviation descriptions
        
    Returns:
        pandas.DataFrame: Original dataframe with 3 new columns added
    """
    # Make a copy of the dataframe to avoid modifying the original
    df_result = df.copy()
    
    # Apply the extraction function to each row of the specified column
    df_result[['pattern_type', 'direction', 'intensity']] = df_result[col_name].apply(
        lambda x: pd.Series(extract_pattern_info(x))
    )
    
    return df_result


In [176]:
df = analyze_deviation_patterns(df)

In [177]:
df

,환자번호,날짜,CMO,MMO,Cap.pal,M.pal,Noise,Loading,Occlusion,OJ/OB,...,Lt,Lateral excursion Protrusive excursion,End feel,CMO_before,CMO_after,MMO_before,MMO_after,pattern_type,direction,intensity
0,2301-01,2023-01-17,34mm --> mm after spray and stretch,"46mm RT M , CAP --> mm after spray and stretch",-,RT M,RT click,-,RT 안닿음,2/2,...,1.24 ->1.74,NaN,soft,34,,46,,other,unspecified,normal
1,2301-01,2023-02-01,38mm --> mm after spray and stretch,"46mm RT M , CAP --> mm after spray and stretch",-,RT) M++ Lt) M+ Temp+,RT click,-,RT 안닿음,2/2,...,1.24 ->1.74,NaN,soft,38,,46,,other,unspecified,normal
2,2301-01,2023-02-17,38mm --> mm after spray and stretch,"46mm RT M , CAP --> mm after spray and stretch",-,RT) M++ Lt) M+ Temp+,RT click,-,RT 안닿음,2/2,...,1.24 ->1.74,NaN,soft,38,,46,,other,unspecified,normal
3,2301-01,2023-03-21,40mm --> mm after spray and stretch,"48mm RT M , CAP --> mm after spray and stretch",-,RT) M+/- Lt) M+/- Temp+/-,RT click,-,RT 안닿음,2/2,...,1.24 ->1.74,NaN,soft,40,,48,,other,unspecified,normal
4,2301-01,2023-04-21,48mm --> mm after spray and stretch,48mm nopain --> 53mm after spray and stretch,-,both) tenderpoint,both) click,-,RT 안닿음,2/2,...,1.24 ->1.74,NaN,soft,48,,48,,other,unspecified,normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28103,2405-86,2024-05-21,20mm --> 53mm after spray and stretch,48mm Lt)M+ --> mm after spray and stretch,-,"Lt)M+(광대부근) , tenderpoint . SCM->T",넘어갈떄 Lt)Click 심함,-,567/567,1/0,...,큰문제는 없으나 약간 두터워진 부분O,NaN,soft,20,53,48,,other,unspecified,normal
28104,2405-88,2024-05-22,mm --> mm after spray and stretch,mm Lt)Cap.M+ --> mm after spray and stretch,Lt)+,"Lt)M++. SCM->T , Rt)M+ ,Both)An T+",Both)click,-,567/567,2/2,...,1.12 ->1.6,NaN,soft,,,,,other,unspecified,normal
28105,2405-89,2024-05-21,29mm --> mm after spray and stretch,43mm Lt cap+--> 55mm after spray and stretch 고착후,Lt)+,Both)M+,Lt)Popping,-,567/567,2/2,...,0.85 ->1.33,NaN,soft,29,,43,,other,unspecified,normal
28106,2405-96,2024-05-29,20mm --> mm after spray and stretch,42mm Rt)+ --> mm after spray and stretch,-,NaN,both) click,-,"4567/4567 (우측 6,7 긴밀하진 않음)",3/4,...,0.74 -> 1.25,NaN,hard,20,,42,,S,right,normal


### Cap.pal & M.pal
씹을때의, 근육의 통증? Cap = 뼈, M 근육

- 스킴
    - 위치 & 강도 & 조건부 상황
    - +,- > + 순으롤 아파짐. (공백은 + 하나)
    - -가 되는 것이 목표



In [181]:
list(set(df['Cap.pal'].unique().tolist()))

['capsule 안쪽 부위 Lt통증호소',
 'Lt)+ 바깥쪽',
 'rt) +감소',
 'Lt capsule pain -> 줄어듬',
 'Lt  +/-',
 'Lt)뻐근',
 'Both)+/-',
 '+/- (Lt cap pain)',
 'RT)++  Lt)+',
 'RT)+,',
 'RT+/-',
 'rt(+)',
 'RT+',
 'Lt pain',
 'Lt) cap 열감,통증 감소',
 'both)+ (입 벌릴때)',
 'both) cap (Rt 더)',
 'Rt) cap+ Lt) +/-',
 'Rt) cap+  (popping 경향성 o)',
 '씹을 떄 rt)capsul',
 '벌릴 때 lt) +',
 'Lt) posterior+',
 '감소했지만 Rt cap pain, cap 뒤쪽 pain',
 'both)+(옆, 뒤쪽)',
 'Lt)cap',
 'both)+ Rt>Lt',
 'rt)+ 통증역치낮음',
 'both + 감소했지만 남아있음.',
 '- 벌릴때아픔',
 'RT CAP ,크게벌리고다물때 LT CAP',
 'BOTH) +RT더',
 nan,
 'rt)cap +/-',
 '벌릴 때 LT)CAP+',
 'both+, RT더',
 '벌릴때 Rt) cap+',
 'Lt cap안쪽',
 'rt)+(옆, 뒤)',
 'Rt) cap+/-, 옆쪽',
 'Rt) 뒤쪽+',
 'LT) CAP',
 'RT)+,아래,뒷부분',
 'LT) 뒤쪽',
 'lt)+(씹을때 lt)+)',
 'Rt)+바깥쪽',
 'Rt) cap -> 벌리때만 pain',
 'Both) cap',
 'Rt)+ , Lt)+/-',
 'Rt)옆,뒤+',
 'RT)+',
 'Lt',
 'Rt) + 세게 누르면 아프다고 하심.',
 'Rt + 뒤쪽',
 'Lt)뒷 +',
 'Rt) +/- Lt) +',
 'Rt) cap +  Lt)+/-',
 'BOTH)+/-',
 'LT CAP 뻐근',
 'Lt) Cap+',
 'Lt)+(꽉물면 pain)',
 'Rt) 뒤쪽+/-',
 'Rt) 옆, 뒤 +',

In [197]:
import re
import pandas as pd
import numpy as np

def classify_capsule_pain(text):
    """
    Capsule pain 텍스트에서 통증 강도, 방향, 상황을 추출하는 함수
    
    Args:
        text (str): Capsule pain 설명 텍스트
        
    Returns:
        tuple: (intensity, direction, situation)
            - intensity (int): 통증 강도 (-1=없음, 0=+/-, 1=+, 2=++, 3=+++)
            - direction (str): 방향 ('left', 'right', 'both', 'unspecified')
            - situation (str): 발생 상황
    """
    # 입력값이 NaN이거나 비어있는 경우 기본값 반환
    if pd.isna(text) or text == '-' or text == '.' or text == '-.' or text == '-,' or text == '':
        return (-1, 'unspecified', '')
    
    # 통증 강도 추출
    intensity = -1  # 기본값: 통증 없음
    
    # 통증 없음 패턴
    if re.search(r'(?:괜찮|감소|없음|거의 없음|n\/s|N\/S|-$|-\.|cap-|감소했|캡술 괜찮음)', text):
        intensity = -1
    # +/- 패턴
    elif re.search(r'(?:\+\/\-|\+\/-|약간|\+,\-|\+-|\-\+)', text):
        intensity = 0
    # +++ 패턴
    elif re.search(r'(?:\+\+\+|\+\+\+\+|\+\+\+)', text):
        intensity = 3
    # ++ 패턴
    elif re.search(r'(?:\+\+|\+\+,|\+\+ )', text):
        intensity = 2
    # + 패턴 (다른 패턴이 없을 경우)
    elif re.search(r'(?:\+|\+,|\+ )', text) and not re.search(r'(?:\+\+|\+\/\-|\+\+\+)', text):
        intensity = 1
    
    # 방향 추출
    direction = 'unspecified'  # 기본값: 방향 미지정
    
    # 양방향 패턴
    if re.search(r'(?i)(?:both|양쪽)', text):
        direction = 'both'
    # 왼쪽 패턴
    elif re.search(r'(?i)(?:lt|왼쪽|LT|Lt|lt\)|Lt\)|LT\))', text) and not re.search(r'(?i)(?:rt|오른|RT|Rt|rt\)|Rt\)|RT\))', text):
        direction = 'left'
    # 오른쪽 패턴
    elif re.search(r'(?i)(?:rt|오른|RT|Rt|rt\)|Rt\)|RT\))', text) and not re.search(r'(?i)(?:lt|왼쪽|LT|Lt|lt\)|Lt\)|LT\))', text):
        direction = 'right'
    # 양쪽 모두 언급된 경우
    elif re.search(r'(?i)(?:lt|왼쪽|LT|Lt)', text) and re.search(r'(?i)(?:rt|오른|RT|Rt)', text):
        direction = 'both'
    
    # 상황 추출
    situation = ''
    
    # 입을 벌릴 때 패턴
    if re.search(r'(?:벌릴|벌리|개구|opening)', text):
        situation = '벌릴 때'
    # 입을 다물 때 패턴
    elif re.search(r'(?:다물|물 때|closing|폐구)', text):
        situation = '다물 때'
    # 씹을 때 패턴
    elif re.search(r'(?:씹을|저작)', text):
        situation = '씹을 때'
    # 누를 때 패턴
    elif re.search(r'(?:누를|누르)', text):
        situation = '누를 때'
    # 움직일 때 패턴
    elif re.search(r'(?:움직|틀)', text):
        situation = '움직일 때'
    # 고착 시 패턴
    elif re.search(r'(?:고착)', text):
        situation = '고착 시'
    
    return (intensity, direction, situation)

def expand_capsule_pain_dataframe(sp, pain_column='Cap.pal'):
    """
    데이터프레임에 통증 강도, 방향, 상황 열을 추가하는 함수
    
    Args:
        sp (pandas.DataFrame): 통증 정보가 포함된 데이터프레임
        pain_column (str): 통증 정보가 저장된 열 이름
        
    Returns:
        pandas.DataFrame: 확장된 데이터프레임
    """
    # 원본 데이터프레임 복사
    sp_result = sp.copy()
    
    # 분류 함수 적용
    classified_data = sp_result[pain_column].apply(classify_capsule_pain)
    
    # 결과 열 추가
    sp_result['Pain_Intensity'] = classified_data.apply(lambda x: x[0])
    sp_result['Pain_Direction'] = classified_data.apply(lambda x: x[1])
    sp_result['Pain_Situation'] = classified_data.apply(lambda x: x[2])
    
    # 통증 강도 매핑 사전 (필요시 표시용)
    intensity_map = {
        -1: '없음',
        0: '+/-',
        1: '+',
        2: '++',
        3: '+++'
    }
    
    sp_result['Pain_Intensity_Label'] = sp_result['Pain_Intensity'].map(intensity_map)
    
    return sp_result

def generate_pain_statistics(sp):
    """
    통증 정보에 대한 통계를 생성하는 함수
    
    Args:
        sp (pandas.DataFrame): 분류된 통증 정보가 포함된 데이터프레임
        
    Returns:
        dict: 통계 정보 딕셔너리
    """
    stats = {
        'total_items': len(sp),
        'intensity_counts': sp['Pain_Intensity'].value_counts().to_dict(),
        'direction_counts': sp['Pain_Direction'].value_counts().to_dict(),
        'situation_counts': sp['Pain_Situation'].value_counts().to_dict(),
        'intensity_by_direction': {},
        'situation_by_direction': {}
    }
    
    # 방향별 강도 분포
    for direction in sp['Pain_Direction'].unique():
        direction_sp = sp[sp['Pain_Direction'] == direction]
        stats['intensity_by_direction'][direction] = direction_sp['Pain_Intensity'].value_counts().to_dict()
    
    # 방향별 상황 분포
    for direction in sp['Pain_Direction'].unique():
        direction_sp = sp[sp['Pain_Direction'] == direction]
        stats['situation_by_direction'][direction] = direction_sp['Pain_Situation'].value_counts().to_dict()
    
    return stats

def print_pain_statistics(stats):
    """
    통증 통계 정보를 출력하는 함수
    
    Args:
        stats (dict): 통계 정보 딕셔너리
    """
    # 강도 매핑
    intensity_map = {
        -1: '없음',
        0: '+/-',
        1: '+',
        2: '++',
        3: '+++'
    }
    
    print(f"총 분석 항목: {stats['total_items']}")
    
    print("\n--- 통증 강도 통계 ---")
    for intensity, count in sorted(stats['intensity_counts'].items()):
        percentage = round(count / stats['total_items'] * 100, 2)
        intensity_label = intensity_map.get(intensity, str(intensity))
        print(f"{intensity_label}: {count}개 ({percentage}%)")
    
    print("\n--- 방향 통계 ---")
    for direction, count in stats['direction_counts'].items():
        percentage = round(count / stats['total_items'] * 100, 2)
        print(f"{direction}: {count}개 ({percentage}%)")
    
    print("\n--- 상황 통계 ---")
    for situation, count in stats['situation_counts'].items():
        if situation == '':
            situation = '미지정'
        percentage = round(count / stats['total_items'] * 100, 2)
        print(f"{situation}: {count}개 ({percentage}%)")
    
    print("\n--- 방향별 강도 분포 ---")
    for direction, intensities in stats['intensity_by_direction'].items():
        print(f"{direction}:")
        direction_total = sum(intensities.values())
        for intensity, count in sorted(intensities.items()):
            percentage = round(count / direction_total * 100, 2)
            intensity_label = intensity_map.get(intensity, str(intensity))
            print(f"  - {intensity_label}: {count}개 ({percentage}%)")

# 예시 사용법
if __name__ == "__main__":
    # 예시 데이터프레임
    data = df['Cap.pal']
    
    sp = pd.DataFrame(data)
    
    # 데이터프레임 확장
    sp_expanded = expand_capsule_pain_dataframe(sp)

    # 결과 확인
    print(sp_expanded[['Cap.pal', 'Pain_Intensity', 'Pain_Direction', 'Pain_Situation', 'Pain_Intensity_Label']])
    
    # 통계 생성 및 출력
    stats = generate_pain_statistics(sp_expanded)
    print_pain_statistics(stats)

      Cap.pal  Pain_Intensity Pain_Direction Pain_Situation  \
0           -              -1    unspecified                  
1           -              -1    unspecified                  
2           -              -1    unspecified                  
3           -              -1    unspecified                  
4           -              -1    unspecified                  
...       ...             ...            ...            ...   
28103       -              -1    unspecified                  
28104    Lt)+               1           left                  
28105    Lt)+               1           left                  
28106       -              -1    unspecified                  
28107       -              -1    unspecified                  

      Pain_Intensity_Label  
0                       없음  
1                       없음  
2                       없음  
3                       없음  
4                       없음  
...                    ...  
28103                   없음  
28104      

In [198]:
sp_expanded

,Cap.pal,Pain_Intensity,Pain_Direction,Pain_Situation,Pain_Intensity_Label
0,-,-1,unspecified,,없음
1,-,-1,unspecified,,없음
2,-,-1,unspecified,,없음
3,-,-1,unspecified,,없음
4,-,-1,unspecified,,없음
...,...,...,...,...,...
28103,-,-1,unspecified,,없음
28104,Lt)+,1,left,,+
28105,Lt)+,1,left,,+
28106,-,-1,unspecified,,없음


In [199]:
df.head()

,환자번호,날짜,CMO,MMO,Cap.pal,M.pal,Noise,Loading,Occlusion,OJ/OB,...,Lt,Lateral excursion Protrusive excursion,End feel,CMO_before,CMO_after,MMO_before,MMO_after,pattern_type,direction,intensity
0,2301-01,2023-01-17,34mm --> mm after spray and stretch,"46mm RT M , CAP --> mm after spray and stretch",-,RT M,RT click,-,RT 안닿음,2/2,...,1.24 ->1.74,NaN,soft,34,,46,,other,unspecified,normal
1,2301-01,2023-02-01,38mm --> mm after spray and stretch,"46mm RT M , CAP --> mm after spray and stretch",-,RT) M++ Lt) M+ Temp+,RT click,-,RT 안닿음,2/2,...,1.24 ->1.74,NaN,soft,38,,46,,other,unspecified,normal
2,2301-01,2023-02-17,38mm --> mm after spray and stretch,"46mm RT M , CAP --> mm after spray and stretch",-,RT) M++ Lt) M+ Temp+,RT click,-,RT 안닿음,2/2,...,1.24 ->1.74,NaN,soft,38,,46,,other,unspecified,normal
3,2301-01,2023-03-21,40mm --> mm after spray and stretch,"48mm RT M , CAP --> mm after spray and stretch",-,RT) M+/- Lt) M+/- Temp+/-,RT click,-,RT 안닿음,2/2,...,1.24 ->1.74,NaN,soft,40,,48,,other,unspecified,normal
4,2301-01,2023-04-21,48mm --> mm after spray and stretch,48mm nopain --> 53mm after spray and stretch,-,both) tenderpoint,both) click,-,RT 안닿음,2/2,...,1.24 ->1.74,NaN,soft,48,,48,,other,unspecified,normal


In [190]:
df.columns

Index(['환자번호', '날짜', 'CMO', 'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading',
       'Occlusion', 'OJ/OB', 'Class', 'Midline Shift', 'Deviation', 'CR-CO',
       'Tongue ridging', 'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt',
       'Lateral excursion Protrusive excursion', 'End feel', 'CMO_before',
       'CMO_after', 'MMO_before', 'MMO_after', 'pattern_type', 'direction',
       'intensity'],
      dtype='object')

In [193]:
df['Cap.pal']

0           -
1           -
2           -
3           -
4           -
         ... 
28103       -
28104    Lt)+
28105    Lt)+
28106       -
28107       -
Name: Cap.pal, Length: 28108, dtype: object

In [191]:
expand_capsule_pain_dataframe(df['Cap.pal'])

KeyError: 'Cap.pal'

### Noise
관절 움직임 시 소음?

- 스킴
    - 위치 & 소음종류(click, snap, pop, etc) & (조건부 상황 or 빈도)
    - popping : 밖에서 들리는 소리, click : 밖에서 안들리는 소리, creptitus : 뼈갈리는 소리
    - creptitus, popping : bad. click : good. bad 에서 good 으로 가는게 목표
    



In [106]:
df[(df['Noise'].notna()) & (df['Noise'] != '-')].sample(20)['Noise']

27943                                      Lt)popping
18374                             -click / popping 가끔
13972           Lt)click ( 환자분께서는 박동성 울리는 소리 들린다고 하심)
3617                                       Rt click작게
20228                                        Lt click
18660                                           click
21073                             Lt click (안나면 걸려있음)
17999                             Lt popping + (저작 시)
24344                  Lt) 덜컥, 환자분 Rt) click 느끼신적 있으심
27114                                -click / popping
24345                  Lt) 덜컥, 환자분 Rt) click 느끼신적 있으심
22185                                     Lt  popping
26715                                   rt) click 약하게
9488                                               --
17486                                    Rt + popping
4044                                         lt click
4345                                        Lt) click
2109                                       BOTH)click
5272                        

### Occlusion
Occlusal ? 교합 혹은 다물었을때 증상? 
, 교합

- 스킴
    - 위치 & 상악 하악의 닿는 이빨 배열? & 특징
- 예시
    - clear
        - none
    - issue
        



In [110]:
df[(df['Occlusion'].notna()) & (df['Occlusion'] != '-')].sample(20)['Occlusion']

14714      4567/4567,  open bite
15710                  4567/4567
27006                    567/567
3384                   Lt 잘 안 닿음
5771     Open bite  both)6,7만 닿음
15504                  4567/4567
21766                 LT456/RT34
26042     both 4567 edge to edge
8710                     567/567
4044                   4567/4567
19306                  4567/4567
32                     4567/4567
17336                 양쪽 4,5 안닿음
26701                  4567/4567
8322                   4657/4567
4660                   4567/4567
8245          Rt7/ Lt76오른쪽 먼저 닿음
25196                  4567/4567
13444                    567/567
25681                        n/s
Name: Occlusion, dtype: object

### OJ/OB
Overbite는 상악(윗니)의 앞니가 하악(아랫니)의 앞니를 덮는 정도를 나타냄 (정상적인 오버바이트는 약 2~4mm 정도가 적당)  \
Overjet는 윗니가 아랫니보다 얼마나 튀어나와 있는지를 측정하는 것 (정상적인 오버젯은 보통 2~3mm)

OB = 2mm 보다 작으면 안좋음. 마이너스가 안좋은거 
OJ = 3mm 이상 커지만 안좋음.

- 스킴
    - 위치 & 강도 & 조건부 상황
- 예시
    - clear
        - none
    - issue
        - 3 / 0 - open bite가능성 
        - 4/3 
        - 3/4
        - 0.5mm/0mm




In [103]:
df[(df['OJ/OB'].notna()) & (df['OJ/OB'] != '-')].sample(20)['OJ/OB']

11980        0/2 open bite 경향
2472                      8/0
10445    3 / 0 - open bite가능성
27714                     2/2
5246                    2.5/4
3040                      4/3
2841                      1/1
8640                      2/2
26868                     3/2
14516                     2/1
148                       2/1
9191                      0/0
26511                     4/3
2216                      3/4
14386                 1mm/1mm
22128                     0/0
19451                     4/3
10522                     2/2
19990               0.5mm/0mm
25614                    3/ 5
Name: OJ/OB, dtype: object

### Class
환자 분류>?

- 스킴
    - 위치 & 강도 & 조건부 상황
- 예시
    - clear
        - none
    - issue
        - s?




In [113]:
df[(df['Class'].notna()) & (df['Class'] != '-')].sample(20)['Class']

9018     1
18523    3
22187    1
2275     2
13518    1
13687    1
4748     2
11666    3
25795    2
8890     3
19161    1
8345     1
19387    1
23509    1
20885    1
21419    1
21054    3
689      1
24174    3
1250     3
Name: Class, dtype: object

### Midline Shift
이빨 중앙 라인에서 왜도

- 스킴
    - 경향성 & 거리
- 예시
    - clear
        - none
    - issue
        - n/s
        - 하악 왼쪽 2mm




In [114]:
df[(df['Midline Shift'].notna()) & (df['Midline Shift'] != '-')].sample(20)['Midline Shift']

5431          하 오 2mm
4748       하악 오른쪽 4mm
4513        상악 왼쪽 2mm
9860              하오2
5414        하악 왼쪽 2mm
24799           하 왼 3
19793     하악 왼쪽으로 2mm
22546      하악왼쪽으로 1mm
16388       하악 왼쪽 0.5
10336             하왼2
13357           하 오 2
11668    하악이 왼쪽으로 1mm
18274             n/s
26536        하악 오른쪽 1
14052       하악 왼쪽 5mm
3418              상왼1
23849       하악 왼쪽 2mm
13019            하오 1
22942         하 왼 1.6
26537         하악 왼쪽 2
Name: Midline Shift, dtype: object

### CR-CO
- Centric Relation
    - Centric Relation은 교합이 가장 안정적이고 균형 잡힌 상태일 때, 즉 두 턱이 제대로 맞물리는 위치를 나타냅니다.
    - CR은 근육과 인대가 최대로 긴장되거나 최적의 위치에 있을 때로, 이 상태에서 하악을 상악과 맞추는 것이 중요합니다.
- Centric Occlusion
    - 실제로 두 턱이 닫힐 때, 즉 치아가 맞물리는 상태를 말합니다. 하악의 치아가 상악의 치아와 접촉하는 지점으로, Centric Occlusion은 교합의 "물어보는" 상태를 의미합니다.
    - CO는 일반적으로 CR과 일치하는 것이 이상적이나, 때로는 CR과 CO가 일치하지 않는 경우도 있을 수 있습니다. 이런 경우에는 교정치료가 필요할 수 있습니다.

어느쪽으로든 2mm 이상이면 안좋다. 두 케이스가 이정도 차이 이상이면 병적으로 의심됨. 방향 상관 없이 다 안좋은거


- 스킴
    - 위치 & 거리
- 예시
    - clear
        - none
    - issue
        - 오른쪽 뒤 0.5-1mm



In [116]:
df[(df['CR-CO'].notna()) & (df['CR-CO'] != '-')].sample(20)['CR-CO']

13001       오른쪽 뒤로 1mm
9802      오른쪽 뒤로 0.5mm
22926       오른쪽 뒤로 1mm
14570             없어보임
9511              거의없음
14688               없음
27709           뒤로 2mm
747                뒤 1
13493            거의 없음
3878          뒤로 1-2mm
15098            0.5mm
15746    오른쪽 뒤 0.5-1mm
7650         오른쪽으로 1mm
14818              1mm
24554     오른쪽 뒤로 1-2mm
12675            1~2MM
11936               없음
19564       우선없는 것 같다.
14456               없음
2280                없음
Name: CR-CO, dtype: object

### Tongue ridging
- 혀의 표면에 나타나는 주름이나 능선을 의미합니다. 이는 보통 혀의 중앙에 세로로 나타나는 주름을 가리키며, 혀의 모양에 영향을 미칠 수 있습니다.

마이너스가 제일 좋다. 

- 스킴
    - 강도
- 예시
    - 강한 양성 (++)	혀의 융기가 매우 뚜렷하고 심각한 수준으로 나타나는 경우
    - 약한 양성 (+)	혀의 융기가 경미하게 관찰되는 상태
    - 음성 (-)	혀의 융기가 전혀 없거나 관찰되지 않는 상태
    - 미확인 (n/s)	데이터 확인이 불가능하거나 판단할 수 없는 상태



In [117]:
df[(df['Tongue ridging'].notna()) & (df['Tongue ridging'] != '-')].sample(20)['Tongue ridging']

909          ++
1769          +
3371          +
22194         +
10769         +
7413          +
26513         +
9502          +
20351         +
27481         +
7180          +
15528         +
7262          +
15565        ++
1173          +
2531          +
25316    약하게 있다
247           +
7479          +
6057         심함
Name: Tongue ridging, dtype: object

### Mucosal ridging
- 구강 점막(즉, 입안의 내부 표면)에 나타나는 주름이나 능선을 의미합니다. 이는 일반적으로 점막이 늘어나거나 두꺼워지면서 생기는 구조적 변화로, 구강 내 다양한 부위에서 발생할 수 있습니다.

마이너스 가 좋다.

- 스킴
    - 강도
- 예시
    - 강한 양성 (++)	혀의 융기가 매우 뚜렷하고 심각한 수준으로 나타나는 경우
    - 약한 양성 (+)	혀의 융기가 경미하게 관찰되는 상태
    - 음성 (-)	혀의 융기가 전혀 없거나 관찰되지 않는 상태
    - 미확인 (n/s)	데이터 확인이 불가능하거나 판단할 수 없는 상태



In [120]:
df[(df['Mucosal ridging'].notna()) & (df['Mucosal ridging'] != '-')].sample(20)['Mucosal ridging']

19603      +
8889       +
1096       +
4768       +
10921      +
26886      +
25868      +
4281       +
18861    n/s
23117      +
20769      +
22053      +
2548       +
23469      +
14366      +
1215       +
26073     심함
10791      +
13283      +
7180       +
Name: Mucosal ridging, dtype: object

### Rt Lt
- 근육 두께
- 스킴
    - 힘 안줬을 떄 초음파로 근육두께 대고, 
    - 화살표 전후로 힘 안 줬을 때, 꽉 물었을 때.
    - 남자는 1.5, 여자는 1.3 미만으로 가는게 목표.
    - 중요한 스키마

In [121]:
df[(df['Rt'].notna()) & (df['Rt'] != '-')].sample(20)['Rt']

22716                0.97 -> 0.98
21777                   1.04->1.3
15757                1.18 -> 1.51
10430                0.97 -> 1.21
12376                0.98 -> 1.24
25299                1.03 -> 1.63
22179               1.13  -> 1.38
9372                 1.07 -> 1.40
1920                  1.14 ->1.57
12261                1.31 -> 1.58
21981                1.07 -> 1.45
8233                  1.30 ->1.73
13361                1.05 -> 1.38
5006                 0.98 -> 1.08
27696                 1.05-> 1.32
3734     1.14 -> 1.56 /0.92->1.21
3781                 0.88 -> 1.28
3662                  1.38 -> 1.9
21475                          ->
703                   1.35 ->1.95
Name: Rt, dtype: object

In [122]:
df[(df['Lt'].notna()) & (df['Lt'] != '-')].sample(20)['Lt']

10150                         1.20 -> 1.74
22605                         0.81 -> 1.29
13911                         1.13 -> 1.68
6059                          1.04 -> 1.42
6395                           1.27 ->1.67
12880                         1.04 -> 1.45
25433                                   ->
25503         1.22 -> 1.57 /       -> 1.33
15840                         0.76 -> 1.22
13790                                   ->
9835                          1.08 -> 1.51
7466     1.15 -> 1.47 /-> 1.23 /0.96->1.21
19183                                   ->
2945                          1.12 -> 1.52
16110                         1.11 -> 1.51
5328           1.2 -> 1.56/->1.33 / ->1.18
26100                                   ->
12753                           1.03->1.23
14409                          1.1 -> 1.43
2248                          1.09 -> 1.42
Name: Lt, dtype: object

### Lateral excursion Protrusive excursion --> 삭제
- Lateral excursion은 하악(아랫니)이 좌우로 움직이는 동작을 말합니다. 즉, 아래턱을 좌우로 한쪽으로 밀었을 때의 움직임을 의미합니다. 이는 턱이 한쪽으로 기울어지는 방향으로 이동하며, 주로 측면 교합을 분석하는 데 사용됩니다.
- Protrusive excursion은 하악이 앞쪽으로 이동하는 동작을 말합니다. 즉, 아래턱을 앞으로 내밀었을 때의 움직임을 의미합니다. 이를 통해 앞쪽 교합을 분석하거나 평가할 수 있습니다.


- 스킴
    - 강도
- 예시
    - issue
        - n/s : 무증상 Ok
        - Lt) PAIN on protrusion, Rt : 왼쪽 특정 위치에 통증, 오른쪽도?



In [71]:
df[(df['Lateral excursion Protrusive excursion'].notna()) & (df['Lateral excursion Protrusive excursion'] != '-')].sample(20)['Lateral excursion Protrusive excursion']

18326            n/s
21508            n/s
18718            n/s
23314            n/s
19023            n/s
18079            n/s
14125    lt crepitus
18787            n/s
18704            n/s
19857            n/s
19848            n/s
14151            wnl
18624            n/s
20040            n/s
14121    lt crepitus
20037            n/s
20092            n/s
17926            n/s
19218            n/s
21424            n/s
Name: Lateral excursion Protrusive excursion, dtype: object

### 치료 계획


- 스킴
    - 치료 종류, 장치 유무, 다음 내방 
    - ck = 체크
    - 몇달 후에 보냐가 중요함. 짧은 시일 내 본다는 것은 잘 안낫고있다는 뜻.



In [132]:
df[(df['치료계획'].notna()) & (df['치료계획'] != '-')].sample(20)['치료계획']

495                           ※ * 가운데 스트레칭, 마사지
3841                                    메모본인 수령
5981                물리치료 , 장치 ck, 근육두께ck [1개월후]
17299    ->교합변화에 있어 고위험군 아니심/ T-SCAN 통해 교합변화 ck
14188                       물리치료 , 장치 ck [1개월후]
3790                  물리치료 , 장치 ck , MMTT [2주후]
19900              물리치료 , 장치 ck , TPI 고려  [1주후]
20515                       물리치료 , 증상 ck [3개월후]
24378               물리치료 , 장치 ck (타원cr t/s 월요일)
22308                        물리치료 , 장치 ck [6주후]
24946                        물리치료 , 장치 ck [2주후]
235                                         냉찜질
6083                               3.차선책으로 MMTT
10335                      물리치료 , APS del [1주후]
26345                       물리치료 , 장치 ck [1개월후]
1638                           ->다음번 APS 중단 가능성
15339                        물리치료 , 혀 drs [1주후]
22175                        물리치료 , 장치 ck [1주후]
2293                   * SS 1일+APS 1일 번갈아가면서 착용
16205                       물리치료 , 장치 ck [1개월후]
Name: 치료계획, dtype: object

In [72]:
df.columns

Index(['환자번호', '날짜', 'CC', '약', '장치 ', '습관', '찜질 ', '마사지, 스트레칭', 'PI', 'CMO',
       'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
       'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
       'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt',
       'Lateral excursion Protrusive excursion', 'End feel', '치료계획',
       'T-scan 악화/개선', 'CBCT 악화/개선', 'CBCT 판독소견'],
      dtype='object')

In [121]:
df[(df['Loading'].notna()) & (df['Loading'] != '-')].sample(20)['Loading']

18380                    n/s
19102              Loading -
25115    LT +click, crepitus
10106                    - -
17785                    n/s
18207                    n/s
19100              Loading -
18115                    n/s
18464                    n/s
18979                    n/s
25121         LT) click 아주약간
17775                    n/s
8144                     교정중
18428                    N/S
18935                    n/s
18664                    n/s
18882                    N/S
18808                    n/s
18523                    n/s
25116    LT +click, crepitus
Name: Loading, dtype: object